In [1]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/iec/MinhHieu/rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.PhysNet import PhysNet_padding_Encoder_Decoder_MAX

In [2]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupC")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/Headmotion/groupC")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 60       # PPG sensor sampling rate (Hz)

# ----- PhysNet preprocessing params -----
CHUNK_LENGTH = 128     # frames per clip
IMG_H, IMG_W = 72, 72  # resolution
LABEL_TYPE   = "DiffNormalized"  # cumsum in post-processing
DATA_FORMAT  = "NCDHW"

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Device: cuda:0
PREPROCESSED_PATH: /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupC
OUTPUT_DIR: /home/iec/MinhHieu/rPPG/results/Headmotion/groupC


In [3]:
# Models list -- uncomment to run additional pretrained models
MODELS = [
    ("PURE_PhysNet",                   "Physnet", "final_model_release/PURE_PhysNet_DiffNormalized.pth"),
    ("SCAMPS_PhysNet",               "Physnet", "final_model_release/SCAMPS_PhysNet_DiffNormalized.pth"),
    ("UBFC-rPPG_PhysNet",            "Physnet", "final_model_release/UBFC-rPPG_PhysNet_DiffNormalized.pth"),
    ("BP4D_PseudoLabel_PhysNet",     "Physnet", "final_model_release/BP4D_PseudoLabel_PhysNet_DiffNormalized.pth"),
    ("MA-UBFC_physnet",              "Physnet", "final_model_release/MA-UBFC_physnet.pth"),
]

# Default to first model
MODEL_NAME, MODEL_TYPE, MODEL_REL_PATH = MODELS[0]
MODEL_PATH = os.path.join(REPO_ROOT, MODEL_REL_PATH)
print(f"Selected model: {MODEL_NAME}")
print(f"Model path: {MODEL_PATH}")

Selected model: PURE_PhysNet
Model path: /home/iec/MinhHieu/rPPG/final_model_release/PURE_PhysNet_DiffNormalized.pth


In [4]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [5]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [6]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [7]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [8]:
# Discover subjects and read ground truth heart rate

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 10 subject folders

  S_001  video=S_001.mkv
  S_002  video=S_002.mkv
  S_003  video=S_003.mkv
  S_004  video=S_004.mkv
  S_005  video=S_005.mkv
  S_006  video=S_006.mkv
  S_007  video=S_007.mkv
  S_008  video=S_008.mkv
  S_009  video=S_009.mkv
  S_010  video=S_010.mkv

Total subjects: 10


In [9]:
# Data preprocessing

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop and resize
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # DiffNormalized data: 3 channels
    diff_data = diff_normalize_data(frames_cropped)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)

    # Chunk into clips of CHUNK_LENGTH frames
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([diff_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])

    # Save per subject
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, H, W, 3)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupC

=== Processing S001 ===
  Video: 2703 frames @ 30 fps
  PPG green: min=1, max=91
  21 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupC/S001

=== Processing S002 ===
  Video: 2703 frames @ 30 fps
  PPG green: min=1, max=94
  21 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupC/S002

=== Processing S003 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=2, max=95
  21 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupC/S003

=== Processing S004 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=1, max=107
  21 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupC/S004

=== Processing S005 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=1, max=117
  21 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupC/S005

=== Processing S006 ===
  Video: 2702 frames @ 30 fps
  PPG green: min=1, max=105
  21 clips -> /home/iec/Min

In [10]:
# PyTorch Dataset + DataLoader

class PhysNetDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (T,)

        # NDHWC -> NCDHW: transpose (3, 0, 1, 2)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = PhysNetDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)
print(f"DataLoader ready: {len(loader)} batches")

Dataset: 210 clips
DataLoader ready: 53 batches


In [11]:
# Post-processing helpers

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=True):
    """Per-subject BVP post-processing with optional cumsum for DiffNormalized."""
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [12]:
# Inference loop -> per-subject results -> aggregate metrics -> export
# Results saved to results_groupC/{model_name}/

# Iterate over all models in the MODELS list
for model_name, model_type, model_rel_path in MODELS:
    model_path = os.path.join(REPO_ROOT, model_rel_path)
    print(f"\n{'='*70}")
    print(f"Model: {model_name}")
    print(f"Path:  {model_path}")
    print(f"{'='*70}")

    # Load model
    model = PhysNet_padding_Encoder_Decoder_MAX(frames=CHUNK_LENGTH)
    
    # Đã thêm weights_only=True để ẩn cảnh báo bảo mật
    state_dict = torch.load(model_path, map_location=DEVICE, weights_only=True)
    
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}
        
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded. Total parameters: {num_params:,}")

    # Run inference
    bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray}
    bvp_labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            data, labels_batch, batch_subjects, batch_chunk_ids = batch

            # data shape: (N, 3, T, H, W) -- already NCDHW from dataset
            data = data.to(DEVICE)

            # PhysNet returns (rPPG, x_visual, x_visual3232, x_visual1616)
            rPPG, _, _, _ = model(data)

            # rPPG shape: (N, T)
            rPPG_np   = rPPG.cpu().numpy()
            labels_np = labels_batch.numpy()

            N = data.shape[0]
            for i in range(N):
                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in bvp_preds_dict:
                    bvp_preds_dict[subj]  = {}
                    bvp_labels_dict[subj] = {}

                bvp_preds_dict[subj][cid]  = rPPG_np[i]
                bvp_labels_dict[subj][cid] = labels_np[i]

    print(f"\nInference complete. Subjects: {sorted(bvp_preds_dict.keys())}")

    # Per-subject results
    per_subject_results = []
    hr_preds_all  = []
    hr_labels_all = []
    snr_all       = []

    FS = VIDEO_FPS

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_label':>10} {'HR_err':>8} {'SNR':>7}")
    print("-" * 55)

    for subj_key in sorted(bvp_preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            bvp_preds_dict[subj_key], bvp_labels_dict[subj_key], fs=FS, diff_flag=True
        )

        snr_db = calculate_snr(pred_processed, hr_label, FS)
        hr_err = hr_pred - hr_label

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "label_heartrate":     hr_label,
            "heartrate_error":     hr_err,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        hr_labels_all.append(hr_label)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {hr_label:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f}")

    hr_preds_all  = np.array(hr_preds_all)
    hr_labels_all = np.array(hr_labels_all)
    snr_all       = np.array(snr_all)

    # Aggregate metrics
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - hr_labels_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(hr_labels_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))
    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))
    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, hr_labels_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\nAggregate Metrics ({model_name}):")
    print(f"  MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"  RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"  MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"  Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"  SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # Export to per-model subdirectory
    model_output_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_output_dir, exist_ok=True)

    metrics_dict = {
        "model":      model_name,
        "n_subjects": n,
        "evaluation_method": "FFT BVP-derived HR",
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,       "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,      "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,      "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
            "SNR":     {"value": mean_snr,  "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":                r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "label_heartrate":     r["label_heartrate"],
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\nMetrics saved to: {json_path}")

    # Export ppg_results.csv
    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":                r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "label_heartrate":     r["label_heartrate"],
            "heartrate_error":     r["heartrate_error"],
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "label_heartrate", "heartrate_error"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)

    print(f"CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

    # Clean up model from GPU
    del model
    torch.cuda.empty_cache()

print(f"\n\nAll models processed. Results in: {OUTPUT_DIR}")


Model: PURE_PhysNet
Path:  /home/iec/MinhHieu/rPPG/final_model_release/PURE_PhysNet_DiffNormalized.pth
Model loaded. Total parameters: 768,577


Inference: 100%|██████████| 53/53 [00:18<00:00,  2.88it/s]



Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          87.891     85.693    2.197   -5.73
S_002          59.766     86.572  -26.807   -5.39
S_003          76.465     76.025    0.439   -2.25
S_004          59.326     97.559  -38.232   -6.68
S_005          87.451     87.451    0.000   -0.53
S_006          61.523     61.963   -0.439    1.08
S_007          99.316     94.922    4.395   -5.53
S_008          56.689     56.689    0.000    0.92
S_009          72.949     73.389   -0.439   -2.75
S_010          74.268     89.209  -14.941   -7.67

Aggregate Metrics (PURE_PhysNet):
  MAE     : 8.7891 +/- 4.0692 bpm
  RMSE    : 15.5829 +/- 12.0574 bpm
  MAPE    : 9.5982 +/- 4.3394 %
  Pearson : 0.4719 +/- 0.3117
  SNR     : -3.4537 +/- 0.9571 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupC/PURE_PhysNe

Inference: 100%|██████████| 53/53 [00:01<00:00, 29.11it/s]



Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          90.088     85.693    4.395   -6.23
S_002          74.268     86.572  -12.305   -7.85
S_003          91.846     76.025   15.820   -5.25
S_004          70.312     97.559  -27.246   -7.25
S_005          78.662     87.451   -8.789   -8.24
S_006          91.846     61.963   29.883   -6.68
S_007          60.205     94.922  -34.717   -6.77
S_008          73.828     56.689   17.139   -4.64
S_009          73.828     73.389    0.439   -5.64
S_010          68.115     89.209  -21.094   -9.55

Aggregate Metrics (SCAMPS_PhysNet):
  MAE     : 17.1826 +/- 3.3545 bpm
  RMSE    : 20.1933 +/- 11.0935 bpm
  MAPE    : 21.7407 +/- 4.4190 %
  Pearson : -0.4660 +/- 0.3128
  SNR     : -6.8106 +/- 0.4434 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupC/SCAMPS

...


Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          85.693     85.693    0.000   -2.56
S_002          86.133     86.572   -0.439    1.42
S_003          78.662     76.025    2.637   -0.92
S_004         102.832     97.559    5.273   -2.34
S_005          89.648     87.451    2.197    3.33
S_006          61.523     61.963   -0.439   -1.17
S_007          99.316     94.922    4.395    1.47
S_008          56.689     56.689    0.000    1.29
S_009          73.389     73.389    0.000   -2.50
S_010          84.375     89.209   -4.834   -3.49

Aggregate Metrics (UBFC-rPPG_PhysNet):
  MAE     : 2.0215 +/- 0.6465 bpm
  RMSE    : 2.8750 +/- 1.8132 bpm
  MAPE    : 2.2651 +/- 0.6914 %
  Pearson : 0.9829 +/- 0.0651
  SNR     : -0.5455 +/- 0.6828 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupC/UBFC-rPP

Inference: 100%|██████████| 53/53 [00:01<00:00, 28.30it/s]



Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          88.770     85.693    3.076   -0.73
S_002          86.133     86.572   -0.439    1.03
S_003          76.025     76.025    0.000    3.46
S_004          97.559     97.559    0.000    0.69
S_005          87.451     87.451    0.000    5.51
S_006          61.523     61.963   -0.439   -2.67
S_007          94.922     94.922    0.000    0.02
S_008          56.689     56.689    0.000    2.79
S_009          70.312     73.389   -3.076   -0.95
S_010          87.451     89.209   -1.758   -3.52

Aggregate Metrics (BP4D_PseudoLabel_PhysNet):
  MAE     : 0.8789 +/- 0.3831 bpm
  RMSE    : 1.4967 +/- 1.0849 bpm
  MAPE    : 1.0969 +/- 0.4803 %
  Pearson : 0.9939 +/- 0.0390
  SNR     : 0.5642 +/- 0.8332 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupC/BP

Inference: 100%|██████████| 53/53 [00:01<00:00, 28.62it/s]



Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          88.770     85.693    3.076    0.24
S_002          86.133     86.572   -0.439    3.74
S_003          76.025     76.025    0.000    0.41
S_004          97.119     97.559   -0.439    0.83
S_005          87.451     87.451    0.000    5.65
S_006          76.025     61.963   14.062   -4.79
S_007          96.680     94.922    1.758    3.34
S_008         115.576     56.689   58.887   -0.17
S_009          72.949     73.389   -0.439   -1.71
S_010          92.285     89.209    3.076   -4.53

Aggregate Metrics (MA-UBFC_physnet):
  MAE     : 8.2178 +/- 5.4888 bpm
  RMSE    : 19.2042 +/- 18.0877 bpm
  MAPE    : 13.7018 +/- 9.7247 %
  Pearson : 0.0186 +/- 0.3535
  SNR     : 0.3017 +/- 1.0168 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupC/MA-UBFC_

In [13]:
# convert metric.json to csv

import os
import glob
import json
import pandas as pd

# 1. Khai báo đường dẫn gốc chứa các thư mục model dựa trên ảnh của bạn
ROOT_DIR = OUTPUT_DIR

# 2. Tìm tất cả các file metrics.json nằm trong các thư mục con
json_files = glob.glob(os.path.join(ROOT_DIR, "*", "metrics.json"))

data_rows = []

# 3. Lặp qua từng file JSON để lấy dữ liệu
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        model_name = data.get("model", "Unknown")
        n_subjects = data.get("n_subjects", 0)
        metrics = data.get("aggregate_metrics", {})
        
        # Lấy giá trị MAE gốc để làm tiêu chí sắp xếp (Rank)
        mae_raw = metrics.get("MAE", {}).get("value", float('inf'))
        
        # Hàm định dạng chữ theo chuẩn "value +/- se" (làm tròn 2 chữ số)
        def format_metric(m):
            if not m: return ""
            return f"{m.get('value', 0):.2f} +/- {m.get('se', 0):.2f}"

        # Đẩy dữ liệu vào 1 hàng (row)
        row = {
            "model": model_name,
            "# subjects": n_subjects,
            "MAE_raw": mae_raw, # Cột tạm để sort
            "MAE (bpm)": format_metric(metrics.get("MAE")),
            "RMSE (bpm)": format_metric(metrics.get("RMSE")),
            "MAPE (%)": format_metric(metrics.get("MAPE")),
            "Pearson": format_metric(metrics.get("Pearson")),
            "SNR (dB)": format_metric(metrics.get("SNR")),
        }
        data_rows.append(row)

# 4. Chuyển thành DataFrame (bảng)
df = pd.DataFrame(data_rows)

if not df.empty:
    # Sắp xếp bảng theo giá trị MAE thô (từ thấp nhất -> cao nhất)
    df = df.sort_values(by="MAE_raw", ascending=True).reset_index(drop=True)
    
    # Thêm cột 'rank' vào vị trí đầu tiên (bắt đầu từ 1)
    df.insert(0, "rank", df.index + 1)
    
    # Xóa cột 'MAE_raw' vì không cần hiển thị ra CSV
    df = df.drop(columns=["MAE_raw"])
    
    # 5. Xuất ra file CSV
    out_csv_path = os.path.join(ROOT_DIR, "Model_Performance_Metrics.csv")
    df.to_csv(out_csv_path, index=False)
    
    print(f"✅ Đã gom thành công {len(json_files)} file JSON!")
    print(f"✅ File tổng hợp được lưu tại:\n{out_csv_path}\n")
    print("Preview dữ liệu:")
    print(df.head().to_string(index=False))
else:
    print("❌ Không tìm thấy file metrics.json nào trong thư mục!")

✅ Đã gom thành công 5 file JSON!
✅ File tổng hợp được lưu tại:
/home/iec/MinhHieu/rPPG/results/Headmotion/groupC/Model_Performance_Metrics.csv

Preview dữ liệu:
 rank                    model  # subjects      MAE (bpm)      RMSE (bpm)       MAPE (%)        Pearson       SNR (dB)
    1 BP4D_PseudoLabel_PhysNet          10  0.88 +/- 0.38   1.50 +/- 1.08  1.10 +/- 0.48  0.99 +/- 0.04  0.56 +/- 0.83
    2        UBFC-rPPG_PhysNet          10  2.02 +/- 0.65   2.87 +/- 1.81  2.27 +/- 0.69  0.98 +/- 0.07 -0.55 +/- 0.68
    3          MA-UBFC_physnet          10  8.22 +/- 5.49 19.20 +/- 18.09 13.70 +/- 9.72  0.02 +/- 0.35  0.30 +/- 1.02
    4             PURE_PhysNet          10  8.79 +/- 4.07 15.58 +/- 12.06  9.60 +/- 4.34  0.47 +/- 0.31 -3.45 +/- 0.96
    5           SCAMPS_PhysNet          10 17.18 +/- 3.35 20.19 +/- 11.09 21.74 +/- 4.42 -0.47 +/- 0.31 -6.81 +/- 0.44
